In [29]:
import pandas as pd
df = pd.read_csv("data\enron_spam_data.csv")

In [30]:
df.head()

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [31]:
df.columns

Index(['Unnamed: 0', 'Subject', 'Message', 'Spam/Ham', 'Date'], dtype='object')

In [32]:
df.shape

(33716, 5)

In [33]:
import pandas as pd

# Assuming you have already loaded your dataframe
# df = pd.read_csv('kaggle_email_data.csv')

print("🚨 Nulls before cleaning:")
print(df.isnull().sum())

# 1. Drop the useless index column
# 'Unnamed: 0' is almost always a leftover row number from whoever saved the CSV.
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

# 2. Drop un-trainable rows
# If the 'Spam/Ham' label is missing, the model can't learn from it. We must drop these.
df.dropna(subset=['Spam/Ham'], inplace=True)

# 3. Smart Fill for Text Columns
# Blank subjects or blank bodies happen in real life. 
# We fill them with empty strings so your TF-IDF vectorizer doesn't crash later.
df['Subject'] = df['Subject'].fillna('')
df['Message'] = df['Message'].fillna('')

# 4. Clean out completely empty emails
# If BOTH the subject and the message are completely blank, it's garbage data. Let's drop it.
df = df[~((df['Subject'] == '') & (df['Message'] == ''))]

# 5. Standardize the target column names (Optional but recommended)
# This makes it match the code you used for the SMS model!
# df.rename(columns={'Spam/Ham': 'target'}, inplace=True)
# df['target'] = df['target'].map({'ham': 0, 'spam': 1})

print("\n✅ Nulls after cleaning:")
print(df.isnull().sum())
print(f"\n📊 Total usable rows remaining: {len(df)}")

🚨 Nulls before cleaning:
Unnamed: 0     0
Subject        0
Message       52
Spam/Ham       0
Date           0
dtype: int64

✅ Nulls after cleaning:
Subject     0
Message     0
Spam/Ham    0
Date        0
dtype: int64

📊 Total usable rows remaining: 33716


In [34]:
df["Spam/Ham"].nunique()

2

In [35]:
df.to_csv("data\enron_spam_data_cleaned.csv")

In [36]:
import pandas as pd

# 1. Load your messy dataframe
# (Replace with your actual filename)
df = pd.read_csv('data\enron_spam_data_cleaned.csv', low_memory=False)

# 2. Define the columns that SHOULD exist
expected_cols = ['Unnamed: 0', 'Subject', 'Message', 'Spam/Ham', 'Date']

# 3. Find all the "extra" columns created by the rogue commas
extra_cols = [col for col in df.columns if col not in expected_cols]

if extra_cols:
    print(f"🚨 Found {len(extra_cols)} fragmented columns. Healing the dataset...")
    
    # 4. Fill NaNs with empty strings so we don't accidentally add the word 'nan' to our emails
    df['Message'] = df['Message'].astype(str).fillna('')
    
    # 5. Loop through every extra column and glue the text back onto the 'Message' column
    for col in extra_cols:
        df[col] = df[col].astype(str).replace('nan', '') # Clean the extra column
        df['Message'] = df['Message'] + " " + df[col]    # Glue it together
        
    # 6. Drop the extra columns now that we've absorbed their data
    df.drop(columns=extra_cols, inplace=True)
    
    # 7. Clean up multiple spaces
    df['Message'] = df['Message'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    print("✅ Fragmented text successfully re-merged!")
else:
    print("✅ No fragmented columns found.")

# Save the healed CSV
df.to_csv('data\cleaned_emails_healed.csv', index=False)

✅ No fragmented columns found.


In [37]:
df.columns

Index(['Unnamed: 0', 'Subject', 'Message', 'Spam/Ham', 'Date'], dtype='object')

final data collection

In [40]:
import pandas as pd
import json
from pathlib import Path
import os

# 1. Define Paths
kaggle_csv_path = Path('data/cleaned_emails_healed.csv')
final_output_path = Path('data/email_final_merged.csv')

# Explicitly define your target folders
json_folders = [
    Path("email_sim_data_05_03_13h_GEN_100_fail"),
    Path("email_sim_data_v2_07_04_23h_GEN_300"),
    Path("email_sim_data_v2_15_04_10h_GEN_1000"),
    Path("email_sim_data_05_03_13h_GEN_100")
]

print("🔄 Phase 1: Processing Kaggle CSV...")
try:
    df_kaggle = pd.read_csv(kaggle_csv_path, engine='python', on_bad_lines='skip')
    
    cols_to_drop = ['Unnamed: 0', 'Date']
    df_kaggle.drop(columns=[col for col in cols_to_drop if col in df_kaggle.columns], inplace=True)
    df_kaggle.rename(columns={'Subject': 'subject', 'Message': 'body', 'Spam/Ham': 'target'}, inplace=True)
    
    df_kaggle['target'] = df_kaggle['target'].map({'ham': 0, 'spam': 1})
    df_kaggle.dropna(subset=['target'], inplace=True)
    print(f"✅ Kaggle data ready. Rows: {len(df_kaggle)}")
except FileNotFoundError:
    print(f"❌ ERROR: Could not find Kaggle CSV at {kaggle_csv_path}. Check your working directory.")
    df_kaggle = pd.DataFrame()


print("\n🔄 Phase 2: Harvesting JSON files from explicit folders...")
json_rows = []
total_json_files = 0

for folder_path in json_folders:
    print(f"\n📂 Scanning folder: {folder_path}")
    
    if not folder_path.exists():
        print(f"   ❌ FOLDER NOT FOUND! Make sure the path is correct relative to where you are running this script.")
        continue
        
    # Look for BOTH .json files and files with no extension just in case
    files_in_folder = list(folder_path.glob("*.json")) + list(folder_path.glob("*.txt"))
    
    folder_file_count = 0
    folder_row_count = 0
    
    for file_path in files_in_folder:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                if not content:
                    continue
                    
                data = json.loads(content)
                
                if isinstance(data, dict):
                    data = [data]
                    
                for item in data:
                    if 'ham' in item:
                        json_rows.append({
                            'subject': item['ham'].get('subject', ''),
                            'body': item['ham'].get('body', ''),
                            'target': 0
                        })
                        folder_row_count += 1
                    if 'spam' in item:
                        json_rows.append({
                            'subject': item['spam'].get('subject', ''),
                            'body': item['spam'].get('body', ''),
                            'target': 1
                        })
                        folder_row_count += 1
                
                folder_file_count += 1
                total_json_files += 1
                
        except json.JSONDecodeError:
            print(f"   ⚠️ Skipping {file_path.name}: Invalid JSON format inside.")
        except Exception as e:
            print(f"   ❌ Error on {file_path.name}: {e}")
            
    print(f"   -> Found and parsed {folder_file_count} files.")
    print(f"   -> Extracted {folder_row_count} new rows from this folder.")

df_json = pd.DataFrame(json_rows)
print(f"\n✅ Total Harvested: {len(df_json)} rows from {total_json_files} files across all folders.")


print("\n🔄 Phase 3: Merging & Cleaning...")
if not df_json.empty and not df_kaggle.empty:
    df_final = pd.concat([df_kaggle, df_json], ignore_index=True)
elif not df_json.empty:
    df_final = df_json
else:
    df_final = df_kaggle

if not df_final.empty:
    df_final['subject'] = df_final['subject'].fillna('')
    df_final['body'] = df_final['body'].fillna('')
    df_final = df_final[~((df_final['subject'] == '') & (df_final['body'] == ''))]

    # Save the final dataset
    final_output_path.parent.mkdir(parents=True, exist_ok=True) # Ensure data folder exists
    df_final.to_csv(final_output_path, index=False, encoding='utf-8')

    print("-" * 40)
    print("🎉 MERGE COMPLETE!")
    print(f"🎯 Final dataset size: {len(df_final)} rows")
    print(f"📁 Saved to: {final_output_path}")
    print("-" * 40)
else:
    print("❌ MERGE FAILED: No data was loaded from either the CSV or the JSON folders.")

🔄 Phase 1: Processing Kaggle CSV...
✅ Kaggle data ready. Rows: 33712

🔄 Phase 2: Harvesting JSON files from explicit folders...

📂 Scanning folder: email_sim_data_05_03_13h_GEN_100_fail
   -> Found and parsed 16 files.
   -> Extracted 32 new rows from this folder.

📂 Scanning folder: email_sim_data_v2_07_04_23h_GEN_300
   -> Found and parsed 266 files.
   -> Extracted 532 new rows from this folder.

📂 Scanning folder: email_sim_data_v2_15_04_10h_GEN_1000
   -> Found and parsed 574 files.
   -> Extracted 1148 new rows from this folder.

📂 Scanning folder: email_sim_data_05_03_13h_GEN_100
   -> Found and parsed 96 files.
   -> Extracted 192 new rows from this folder.

✅ Total Harvested: 1904 rows from 952 files across all folders.

🔄 Phase 3: Merging & Cleaning...
----------------------------------------
🎉 MERGE COMPLETE!
🎯 Final dataset size: 35616 rows
📁 Saved to: data\email_final_merged.csv
----------------------------------------


In [41]:
df_final.shape

(35616, 3)